In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import joblib
import os
from datetime import datetime

# Preprocessing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_percentage_error, make_scorer

# Models
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor
from openpyxl import load_workbook
from lightgbm import LGBMRegressor

# Model selection
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    KFold,
)

# Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Feature importance
from sklearn.inspection import permutation_importance

# Logging
from openpyxl.styles import Font, Alignment
from openpyxl.formatting.rule import ColorScaleRule
from sklearn.model_selection import ParameterGrid


In [2]:
# =============================================================================
# SECTION 1: LOAD DATA
# =============================================================================

CONFIG = {
    "input": "Outputs/clean_property_data.parquet",
    "test_size": 0.2,
    "random_state": 42,
    "cv_folds": 5,
    "n_jobs": 1,  # keep at 1 to avoid memory crashes on large data
}

print("Loading dataset...")
df = pd.read_parquet(CONFIG["input"])
print(f"  Rows: {len(df):,}, Columns: {len(df.columns)}")
print(f"  Columns: {df.columns.tolist()}")

df = df.sample(n=10_000, random_state=42) 

Loading dataset...
  Rows: 4,295,153, Columns: 28
  Columns: ['price', 'property_type_x', 'new_build', 'duration', 'total_floor_area', 'current_energy_efficiency', 'current_energy_rating', 'number_habitable_rooms', 'tenure', 'construction_age_band', 'built_form', 'main_fuel', 'exact_lat', 'exact_lon', 'dist_primary_km', 'dist_secondary_km', 'dist_rail_km', 'rail_within_1km', 'rail_within_5km', 'dist_metro_km', 'metro_within_1km', 'dist_airport_km', 'dist_coast_km', 'dist_town_km', 'construction_year', 'construction_year_exact', 'property_age', 'sale_time']


In [3]:
# =============================================================================
# SECTION 2: DEFINE FEATURES AND TARGET
# =============================================================================

target = "price"

numeric_features = [
    # EPC features
    "total_floor_area",           # floor area in m²
    "current_energy_efficiency",  # numeric efficiency score
    "number_habitable_rooms",     # number of rooms

    # Location — exact coordinates
    "exact_lat",                  # property latitude
    "exact_lon",                  # property longitude

    # Spatial — school distances
    "dist_primary_km",            # distance to nearest primary school
    "dist_secondary_km",          # distance to nearest secondary school

    # Spatial — transport distances and density
    "dist_rail_km",               # distance to nearest rail station
    "rail_within_1km",            # number of rail stations within 1km
    "rail_within_5km",            # number of rail stations within 5km
    "dist_metro_km",              # distance to nearest metro/underground
    "metro_within_1km",           # number of metro stations within 1km
    "dist_airport_km",            # distance to nearest airport

    # Spatial — geography
    "dist_coast_km",              # distance to nearest coastline
    "dist_town_km",               # distance to nearest major town (ONS 75k+)

    # Date
    #"sale_year",                  # year of sale
    #"sale_month",                 # month of sale
    "sale_time",

    # Construction
    "construction_year",          # exact year of construction, where available
    "construction_year_exact",    # 1 if exact year was provided, 0 otherwise
    "property_age",               # property age at the time of sale
]

categorical_features = [
    # Land Registry
    "property_type_x",            # D=Detached, S=Semi, T=Terraced, F=Flat
    "new_build",                  # Y/N
    "duration",                   # F=Freehold, L=Leasehold

    # EPC
    "current_energy_rating",      # A-G energy rating
    "tenure",                     # owner-occupied, rented, etc.
    "built_form",                 # Detached, Semi-Detached, Mid-Terrace, etc.
    "construction_age_band",      # e.g. 1900-1929, 1967-1975
    "main_fuel",                  # mains gas, electricity, etc.
]

# Removed from previous version:
# - property_type_y: overlaps with property_type_x and built_form
# - transaction_type: describes why EPC was created, not a property attribute
# - mains_gas_flag: 43% NaN, redundant with main_fuel
# - binary_features: new_build moved to categorical (Y/N works with OneHot)

all_features = numeric_features + categorical_features

# Verify all features exist
missing = [f for f in all_features if f not in df.columns]
if missing:
    print(f"  WARNING: missing columns: {missing}")
    numeric_features = [f for f in numeric_features if f in df.columns]
    categorical_features = [f for f in categorical_features if f in df.columns]
    all_features = numeric_features + categorical_features

X = df[all_features]
y = df[target]

print(f"\n  Features: {len(all_features)}")
print(f"    Numeric:     {len(numeric_features)}")
print(f"    Categorical: {len(categorical_features)}")
print(f"  Target: {target} (mean=£{y.mean():,.0f}, median=£{y.median():,.0f})")


  Features: 27
    Numeric:     19
    Categorical: 8
  Target: price (mean=£325,862, median=£268,000)


In [4]:
# =============================================================================
# SECTION 3: TRAIN/TEST SPLIT
# =============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=CONFIG["test_size"],
    random_state=CONFIG["random_state"],
)

print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")

  X_train: (8000, 27)
  X_test:  (2000, 27)


In [5]:
# =============================================================================
# SECTION 4: PREPROCESSOR
# =============================================================================

# Numeric: fill any remaining NaN with median, then scale
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Categorical: fill any remaining NaN with "Unknown", then one-hot encode
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [6]:
# =============================================================================
# SECTION 5: BUILD PIPELINES
# =============================================================================

pipe_ridge = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Ridge())
])

pipe_lasso = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Lasso(max_iter=1000))  # up to 5000 for final training
])

pipe_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(random_state=42, n_jobs=-1))
])

pipe_xgb = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(random_state=42, n_jobs=-1, tree_method="hist", device="cuda"))
])

pipe_mlp = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", MLPRegressor(random_state=42, max_iter=500, early_stopping=True))
])

pipe_lgbm = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1,))
])

In [7]:
# =============================================================================
# SECTION 6: PARAM GRIDS
# =============================================================================

param_grids = {
    "Ridge": {
        "pipeline": pipe_ridge,
        "params": {
            "regressor__alpha": [1.0, 10.0, 100.0, 150.0, 200.0],
        }
    },
    "XGBoost": {
        "pipeline": pipe_xgb,
        "params": { 
            "regressor__n_estimators": [200, 300, 500], 
            "regressor__max_depth": [4, 6, 8, 10], 
            "regressor__learning_rate": [0.05, 0.1], 
            "regressor__subsample": [0.8],
            "regressor__colsample_bytree": [0.8], 
            "regressor__min_child_weight": [10, 50],
        },
    },
    "LightGBM": {
        "pipeline": pipe_lgbm,
        "params": {
            "regressor__n_estimators": [500, 700],
            "regressor__max_depth": [8, 10, 12],
            "regressor__learning_rate": [0.05, 0.1],
            "regressor__num_leaves": [63, 127],
        },
    },
    "RandomForest": {
        "pipeline": pipe_rf,
        "params": {
            "regressor__n_estimators": [100, 200],
            "regressor__max_depth": [10, 20, None],
            "regressor__min_samples_leaf": [5, 10],
        }
    },
}

In [8]:
# =============================================================================
# SECTION 7: QUICK TEST — Ridge baseline without GridSearch
# =============================================================================

pipe_quick = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Ridge(alpha=100.0))
])

pipe_quick.fit(X_train, y_train)
print(f"Quick Ridge baseline:")
print(f"  Train R²: {pipe_quick.score(X_train, y_train):.4f}")
print(f"  Test R²:  {pipe_quick.score(X_test, y_test):.4f}")

Quick Ridge baseline:
  Train R²: 0.6223
  Test R²:  0.5826


In [9]:
def log_experiment(log_file, log_entry):
    csv_file = log_file.replace(".html", ".csv")

    if os.path.exists(csv_file):
        log_df = pd.concat([pd.read_csv(csv_file), pd.DataFrame([log_entry])], ignore_index=True)
    else:
        log_df = pd.DataFrame([log_entry])

    # Save raw data for analysis
    log_df.to_csv(csv_file, index=False)

    # Format copy for HTML display
    display_df = log_df.copy()

    for col in ["Train R²", "Test R²"]:
        if col in display_df: display_df[col] = display_df[col].map(lambda x: f"{x:.4f}" if pd.notna(x) else "—")

    for col in ["Train RMSE (£)", "Test RMSE (£)", "Train MAE (£)", "Test MAE (£)"]:
        if col in display_df: display_df[col] = display_df[col].map(lambda x: f"£{x:,.0f}" if pd.notna(x) else "—")

    for col in ["Train MAPE (%)", "Test MAPE (%)"]:
        if col in display_df: display_df[col] = display_df[col].map(lambda x: f"{x:.1f}%" if pd.notna(x) else "—")

    if "Time (s)" in display_df:
        display_df["Time (s)"] = display_df["Time (s)"].map(lambda x: f"{x:.1f}s" if pd.notna(x) else "—")

    table_html = display_df.to_html(index=False, escape=True, classes="experiment-table")

    html = f"""<!DOCTYPE html><html><head><meta charset="UTF-8"><title>Model Experiment Log</title>
<style>
body{{font-family:Arial,sans-serif;margin:30px;background:#f5f5f5}}
h1{{margin-bottom:20px}}
.table-container{{overflow-x:auto;background:white;padding:20px;border-radius:8px}}
table{{border-collapse:collapse;width:100%;min-width:1600px;font-size:14px}}
th{{background:#333;color:white;padding:10px;text-align:center;position:sticky;top:0}}
td{{padding:8px;border:1px solid #ddd;text-align:center;vertical-align:top}}
tr:nth-child(even){{background:#f7f7f7}}
td:nth-child(16){{text-align:left;max-width:600px;white-space:normal;word-wrap:break-word}}
</style></head><body><h1>Model Experiment Log</h1>
<p>Experiments recorded: <strong>{len(display_df)}</strong></p>
<div class="table-container">{table_html}</div></body></html>"""

    with open(log_file, "w", encoding="utf-8") as file:
        file.write(html)

    print(f"Logged HTML: {log_file}")
    print(f"Logged CSV:  {csv_file}")

In [ ]:
from sklearn.model_selection import ParameterGrid
from tqdm.notebook import tqdm

cv = KFold(
    n_splits=CONFIG["cv_folds"],
    shuffle=True,
    random_state=CONFIG["random_state"],
)

log_file = "Outputs/experiment_log_file.html"
results = {}
score_name = 'MAPE'

for name, config in param_grids.items():
    param_list = list(ParameterGrid(config["params"]))
    total_fits = len(param_list) * CONFIG["cv_folds"]
    
    print(f"\n{'='*50}")
    print(f"Training {name}... ({len(param_list)} combos × {CONFIG['cv_folds']} folds)")
    print(f"{'='*50}")
    
    start = time.time()
    best_score = float("-inf")
    best_params = None
    
    pbar = tqdm(total=total_fits, desc=name, unit="fit")
    
    for params in param_list:
        pipe = config["pipeline"]
        pipe.set_params(**params)
        
        fold_scores = []
        for train_idx, val_idx in cv.split(X_train):
            X_fold_train = X_train.iloc[train_idx]
            y_fold_train = y_train.iloc[train_idx]
            X_fold_val = X_train.iloc[val_idx]
            y_fold_val = y_train.iloc[val_idx]
            
            pipe.fit(X_fold_train, y_fold_train)
            y_pred = pipe.predict(X_fold_val)
            if score_name == 'MAPE':
                score = -mean_absolute_percentage_error(y_fold_val, y_pred)
            else:
                score = -mean_absolute_error(y_fold_val, y_pred)
            fold_scores.append(score)
            pbar.update(1)
        
        avg_score = np.mean(fold_scores)
        if avg_score > best_score:
            best_score = avg_score
            best_params = params
    
    pbar.close()
    
    # Refit best on full training data
    pipe = config["pipeline"]
    pipe.set_params(**best_params)
    pipe.fit(X_train, y_train)
    
    train_time = time.time() - start
    
    y_pred_train = pipe.predict(X_train)
    y_pred_test = pipe.predict(X_test)
    
    results[name] = {
        "best_params": best_params,
        "train_time": train_time,
        "train_rmse": np.sqrt(mean_squared_error(y_train, y_pred_train)),
        "test_rmse": np.sqrt(mean_squared_error(y_test, y_pred_test)),
        "train_mae": mean_absolute_error(y_train, y_pred_train),
        "test_mae": mean_absolute_error(y_test, y_pred_test),
        "train_r2": r2_score(y_train, y_pred_train),
        "test_r2": r2_score(y_test, y_pred_test),
        "model": pipe,
        "train_mape": np.mean(np.abs((y_train - y_pred_train) / y_train)) * 100,
        "test_mape": np.mean(np.abs((y_test - y_pred_test) / y_test)) * 100,
    }
    
    r = results[name]
    print(f"\n  Best params: {r['best_params']}")
    print(f"  Time: {r['train_time']:.0f}s")
    print(
        f"  Train — RMSE: £{r['train_rmse']:,.0f}, "
        f"MAE: £{r['train_mae']:,.0f}, "
        f"R²: {r['train_r2']:.4f}, "
        f"MAPE: {r['train_mape']:.1f}%"
    )
    print(
        f"  Test  — RMSE: £{r['test_rmse']:,.0f}, "
        f"MAE: £{r['test_mae']:,.0f}, "
        f"R²: {r['test_r2']:.4f}, "
        f"MAPE: {r['test_mape']:.1f}%"
    )

    log_entry = {
        "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "Model": name,
        "Rows": len(df),
        "Train Rows": len(X_train),
        "Test Rows": len(X_test),
        "Features": len(all_features),
        "Train R²": r["train_r2"],
        "Test R²": r["test_r2"],
        "Train RMSE (£)": r["train_rmse"],
        "Test RMSE (£)": r["test_rmse"],
        "Train MAE (£)": r["train_mae"],
        "Test MAE (£)": r["test_mae"],
        "Train MAPE (%)": r["train_mape"],
        "Test MAPE (%)": r["test_mape"],
        "Time (s)": round(r["train_time"], 1),
        "Best Parameters": str(r["best_params"]),
        "Scoring": score_name,
        "Notes": "",
    }

    log_experiment(log_file, log_entry)


Training Ridge... (5 combos × 5 folds)


Ridge:   0%|          | 0/25 [00:00<?, ?fit/s]


  Best params: {'regressor__alpha': 200.0}
  Time: 2s
  Train — RMSE: £161,540, MAE: £92,186, R²: 0.6215, MAPE: 35.6%
  Test  — RMSE: £172,457, MAE: £92,163, R²: 0.5817, MAPE: 35.7%
Logged HTML: Outputs/experiment_log_file.html
Logged CSV:  Outputs/experiment_log_file.csv

Training XGBoost... (48 combos × 5 folds)


XGBoost:   0%|          | 0/240 [00:00<?, ?fit/s]

RuntimeError: Failed to find CUDA headers. Please install CUDA toolkit headers (e.g., pip install cupy-cuda12x[ctk]) or specify CUDA_PATH environment variable.